# Rebuilding Netflix's MAPS: multimodal artwork personalization on MovieLens

Netflix published [MAPS](https://netflixtechblog.com/maps-netflixs-multimodal-asset-personalization-at-scale-32f96320785e) (Aug 2026): feeding CLIP image embeddings into the artwork ranker so personalization works on brand-new assets (**cold-start**), and consolidating five per-canvas models into one.

This notebook rebuilds the mechanics on public data. Be clear about what is what:

**Real:** MovieLens ratings (~890k), real movie posters, real CLIP embeddings, the model architecture, the IPS estimator, the linear-probe gate.

**Simulated:** Netflix logs artwork impressions and clicks. No public dataset has that, so impressions and clicks are simulated from stated assumptions (numbered below). The point is to show the *mechanics* — how content embeddings fix cold-start, how IPS evaluates a policy offline — not to prove a real-world lift. Netflix's proof is their A/B test; ours is a controlled simulation.

**Self-contained:** data downloads inside the notebook (one parquet from HuggingFace + poster images). No datasets to attach. Runs on Kaggle/Colab with internet on; GPU recommended (CPU works, CLIP takes ~8 min at the 600-movie cap).

Credit: the ideas belong to the Netflix authors (Kong, Deshpande, Abbasi, Yan, Fagnan, Rastogi, Patel, Zhang). This is an independent educational rebuild.


In [ ]:
# all imports in one cell
!pip -q install open_clip_torch

import os, io, math, json, time
import concurrent.futures as cf
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
N_MOVIES_CAP = 600   # raise to 1500+ on GPU if you want
print(device)

## 1. Data: ratings + posters

One parquet file (MovieLens ratings joined with poster URLs, mirrored on HuggingFace), then the poster images themselves. We keep the most-rated movies and drop any whose poster URL is dead.

In [ ]:
PARQUET_URL = ("https://huggingface.co/datasets/ashraq/movielens_ratings/"
               "resolve/main/data/train-00000-of-00001-8c8c7645a52d95e5.parquet")
if not os.path.exists("ratings.parquet"):
    urllib.request.urlretrieve(PARQUET_URL, "ratings.parquet")

df = pd.read_parquet("ratings.parquet").rename(columns={"movie_id": "movieId", "user_id": "userId"})
counts = df.groupby("movieId").size().sort_values(ascending=False)
df = df[df.movieId.isin(counts.index[:N_MOVIES_CAP])]
movies = df.drop_duplicates("movieId")[["movieId", "title", "posters"]].reset_index(drop=True)
print(f"{len(movies)} movies, {len(df)} ratings, {df.userId.nunique()} users")

In [ ]:
# download posters (threaded); drop movies whose poster can't be fetched
os.makedirs("posters", exist_ok=True)

def fetch(row):
    p = f"posters/{row.movieId}.jpg"
    if os.path.exists(p): return row.movieId, True
    try:
        req = urllib.request.Request(row.posters, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=20) as r:
            data = r.read()
        Image.open(io.BytesIO(data)).convert("RGB").save(p)   # validate it's an image
        return row.movieId, True
    except Exception:
        return row.movieId, False

with cf.ThreadPoolExecutor(8) as ex:
    res = list(ex.map(fetch, movies.itertuples()))
ok = {m for m, s in res if s}
movies = movies[movies.movieId.isin(ok)].reset_index(drop=True)
df = df[df.movieId.isin(ok)]
print(f"{len(movies)} movies with posters")

## 2. Manufacture the assets: artwork variants × canvases

Netflix has several artworks per title, each rendered on five canvases (billboard, vertical, landscape, short-panel, panel). We have one poster per movie, so:

- **4 artwork variants** = crops showing different parts of the poster (top, center, bottom, zoom). An imperfect stand-in for real alternative artworks, but the crops do show different content (a face vs title text vs a scene), so their CLIP embeddings differ.
- **5 canvases** = aspect-ratio renders of each variant. Near-duplicates on purpose — that's Netflix's crop-invariance argument, and it's what lets one model pool signal across canvases.

Asset = (movie, variant, canvas) → ~11,000 assets at the 600-movie cap.

In [ ]:
CANVASES = {"billboard": (16, 6), "vertical": (2, 3), "landscape": (4, 3),
            "short_panel": (2, 1), "panel": (1, 1)}
VARIANTS = ["top", "center", "bottom", "zoom"]

def variant_crop(img, v):
    # crop a region of the poster -> a distinct "artwork"
    w, h = img.size
    box = {"top": (0, 0, w, int(h*0.45)), "center": (0, int(h*0.25), w, int(h*0.75)),
           "bottom": (0, int(h*0.55), w, h),
           "zoom": (int(w*0.2), int(h*0.15), int(w*0.8), int(h*0.6))}[v]
    return img.crop(box)

def canvas_render(img, aw, ah, out_w=224):
    # center-crop the artwork to the canvas aspect ratio, then resize
    w, h = img.size; target = aw / ah
    if w / h > target:
        nw = int(h * target); box = ((w-nw)//2, 0, (w+nw)//2, h)
    else:
        nh = int(w / target); box = (0, (h-nh)//2, w, (h+nh)//2)
    return img.crop(box).resize((out_w, max(1, int(out_w / target))))

rows, asset_imgs = [], []
aid = 0
for _, r in movies.iterrows():
    img = Image.open(f"posters/{r.movieId}.jpg").convert("RGB")
    for vi, v in enumerate(VARIANTS):
        art = variant_crop(img, v)
        for ci, (c, (aw, ah)) in enumerate(CANVASES.items()):
            rows.append((aid, r.movieId, vi, ci, c))
            asset_imgs.append(canvas_render(art, aw, ah))
            aid += 1
assets = pd.DataFrame(rows, columns=["asset_id", "movieId", "variant", "canvas", "canvas_name"])
print(len(assets), "assets")

In [ ]:
# look at one movie's assets: 4 variants x 5 canvases
mid = movies.movieId.iloc[0]
sub = assets[assets.movieId == mid]
fig, axes = plt.subplots(4, 5, figsize=(12, 9))
for ax, (_, a) in zip(axes.ravel(), sub.iterrows()):
    ax.imshow(asset_imgs[a.asset_id]); ax.axis("off")
    ax.set_title(f"{VARIANTS[a.variant]} / {a.canvas_name}", fontsize=7)
plt.tight_layout(); plt.show()

## 3. CLIP embeddings

One vector per asset, from pretrained CLIP (ViT-B/32). Netflix's pattern: encode once, store, serve everywhere (their Embedding Store). Here the numpy file is our tiny embedding store — cached and keyed to the asset list, so reruns skip this step.

In [ ]:
import open_clip
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_model = clip_model.to(device).eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")

@torch.no_grad()
def embed_images(imgs, bs=256):
    out = []
    for i in range(0, len(imgs), bs):
        x = torch.stack([preprocess(im) for im in imgs[i:i+bs]]).to(device)
        e = clip_model.encode_image(x)
        out.append((e / e.norm(dim=-1, keepdim=True)).cpu().numpy())
    return np.vstack(out)

# cache keyed to the exact asset list (poster availability can vary run to run)
cache_ok = False
if os.path.exists("clip_emb.npy") and os.path.exists("clip_ids.npy"):
    ids = np.load("clip_ids.npy")
    if len(ids) == len(assets) and (ids == assets.movieId.values).all():
        clip_emb = np.load("clip_emb.npy"); cache_ok = True
if not cache_ok:
    clip_emb = embed_images(asset_imgs).astype(np.float32)
    np.save("clip_emb.npy", clip_emb)
    np.save("clip_ids.npy", assets.movieId.values)
D_CLIP = clip_emb.shape[1]
print(clip_emb.shape)

In [ ]:
# sanity check = Netflix's crop-invariance claim:
# the 5 canvas renders of one variant should be much closer to each other
# than to other variants, and far from other movies
a = assets
sv, sm, rp = [], [], []
for mid in movies.movieId.sample(50, random_state=SEED):
    g = a[a.movieId == mid]; e = clip_emb[g.asset_id.values]
    for v in range(4):
        ev = clip_emb[g[g.variant == v].asset_id.values]
        sv.append(float((ev @ ev.T).mean()))
    sm.append(float((e @ e.T).mean()))
    other = a[a.movieId != mid].sample(20, random_state=SEED)
    rp.append(float((e @ clip_emb[other.asset_id.values].T).mean()))
print(f"same variant, diff canvas: {np.mean(sv):.3f}")   # ~0.78 - crop invariance
print(f"same movie, diff variant : {np.mean(sm):.3f}")   # ~0.56 - variants differ
print(f"different movies         : {np.mean(rp):.3f}")   # ~0.28 - note: NOT ~0")

**Verified numbers (our run):** same variant across canvases **0.78**, same movie across variants **0.56**, different movies **0.28**.

Two lessons sit in that last number. First, crop invariance holds (0.78 ≫ 0.56): one model can pool canvases. Second, cosine between *unrelated* movies is 0.28, not 0 — raw CLIP vectors all share a common direction (the "cone effect"). That matters for the next section.

## 4. Simulate the impression logs

Netflix's logs are private, so we simulate them — every assumption stated and numbered:

- **Assumption 1 — member taste lives in centered CLIP space.** Because of the cone effect above, raw cosines barely separate a movie's variants. So the ground truth uses the **centered** space: subtract the mean embedding, renormalize. A user's taste vector = mean centered embedding of posters of movies they actually rated ≥ 4 (their real MovieLens ratings). Taste for an asset = cosine(user vector, centered asset). The *model* still consumes raw CLIP features — only the simulator is centered. (Our first run skipped centering: every variant got near-identical click probability, and all models scored ≈ random. Centering is the fix.)
- **Assumption 2 — some artworks just work.** A global appeal term: cosine(centered asset, mean centered embedding of the 100 top-rated movies' posters). This is what the linear probe will try to recover.
- **Assumption 3 — click probability.** `p = sigmoid(8·taste + 4·appeal + canvas_bias − 1.5)`. Calibrated on the real embeddings so overall CTR lands ~10–25% and, per (user, movie), the best variant gets p≈0.20 vs p≈0.06 for the worst — a real but noisy signal.
- **Assumption 4 — interaction types.** A positive is a click (70%), a play (25%) or a finish (5%), with long-term reward weights 0.3 / 1.0 / 2.0 (stand-in for Netflix's long-term reward model).
- **Assumption 5 — canvas volumes are skewed.** billboard 40k, vertical 20k, landscape 10k, panel 4k, short_panel 2k impressions. Sparse canvases are where consolidation should help.
- **Assumption 6 — exploration slice.** The logging policy picks uniformly among a movie's 4 variants, so every impression's propensity is exactly 1/4, logged at serving time. (Netflix logs exact propensities on a randomized slice; we make the whole log randomized for simplicity.)
- **20% of movies are held out as cold-start**: their impressions never enter training, only evaluation.

**Known limitation:** ground-truth taste is built from CLIP embeddings and the winning model uses CLIP features, so the simulation favors it by construction. It demonstrates the mechanism, not a real-world effect size.

In [ ]:
# centered space for the GROUND TRUTH (Assumption 1); model still gets raw CLIP
mu = clip_emb.mean(0)
cen = clip_emb - mu
cen = (cen / np.linalg.norm(cen, axis=1, keepdims=True)).astype(np.float32)

first = assets[(assets.canvas == 1) & (assets.variant == 1)]   # ~ the poster itself
poster_cen = dict(zip(first.movieId, cen[first.asset_id.values]))

liked = df[df.rating >= 4.0]
user_vec = {}
for u, g in liked.groupby("userId"):
    if len(g) >= 5:
        v = np.mean([poster_cen[m] for m in g.movieId], axis=0)
        user_vec[u] = v / np.linalg.norm(v)
users = list(user_vec.keys())
U = np.stack([user_vec[u] for u in users]).astype(np.float32)
print(f"{len(users)} users with taste vectors")

# global appeal direction (Assumption 2)
top = df.groupby("movieId").rating.mean().nlargest(100).index
appeal = np.mean([poster_cen[m] for m in top if m in poster_cen], axis=0)
appeal = (appeal / np.linalg.norm(appeal)).astype(np.float32)

CANVAS_BIAS = {0: 0.3, 1: 0.0, 2: 0.0, 3: -0.2, 4: -0.1}
TASTE_ALL = U @ cen.T        # (n_users, n_assets), precomputed
APPEAL_ALL = cen @ appeal

def p_click(u_idx, asset_id, canvas):
    s = 8.0 * TASTE_ALL[u_idx, asset_id] + 4.0 * APPEAL_ALL[asset_id] + CANVAS_BIAS[canvas] - 1.5
    return 1 / (1 + math.exp(-s))   # Assumption 3

In [ ]:
# cold-start split + impression logging (Assumptions 4-6)
mids = movies.movieId.values
perm = rng.permutation(mids)
cold = set(perm[: int(0.2 * len(mids))])
warm = np.array([m for m in mids if m not in cold])
print(f"{len(warm)} warm movies (training), {len(cold)} cold movies (eval only)")

K = len(VARIANTS)
CANVAS_VOL = {0: 40000, 1: 20000, 2: 10000, 3: 2000, 4: 4000}   # Assumption 5
ITYPES = np.array(["click", "play", "finish"]); IPROB = [0.70, 0.25, 0.05]
RHO = {"click": 0.3, "play": 1.0, "finish": 2.0}                # Assumption 4
cand = assets.set_index(["movieId", "variant", "canvas"]).asset_id   # fast lookup

def log_impressions(movie_pool, vol):
    recs = []
    for ci in CANVAS_VOL:
        n = vol[ci]
        us = rng.integers(0, len(users), n)
        ms = rng.choice(movie_pool, n)
        ks = rng.integers(0, K, n)              # uniform among variants
        for u, m, k in zip(us, ms, ks):
            aidx = cand.loc[(m, k, ci)]
            r = int(rng.random() < p_click(u, aidx, ci))
            it = rng.choice(ITYPES, p=IPROB) if r else "none"
            recs.append((u, m, k, ci, aidx, r, it, 1.0 / K))    # Assumption 6
    return pd.DataFrame(recs, columns=["user", "movieId", "variant", "canvas",
                                       "asset_id", "reward", "itype", "prop"])

train_log = log_impressions(warm, CANVAS_VOL)
explore_cold = log_impressions(np.array(list(cold)), {c: 4000 for c in CANVAS_VOL})
explore_warm = log_impressions(warm, {c: 4000 for c in CANVAS_VOL})
print(f"train {len(train_log)} impressions, CTR {train_log.reward.mean():.3f}")

## 5. The model: ID embedding ⊕ CLIP → MLP

Netflix's architecture, miniaturized. Asset representation `h_a = MLP([e_id(a); e_clip(a)])`, scored against a learned user embedding by dot product. Training examples are weighted by the long-term reward of their interaction type (reward-based weighting — how Netflix mixes five canvases of data without hand-tuned weights).

In [ ]:
class AssetModel(nn.Module):
    # use_clip=False -> the ID-only baseline (what Netflix had before MAPS)
    def __init__(self, n_users, n_assets, use_clip, d_id=16, d_h=64):
        super().__init__()
        self.use_clip = use_clip
        self.u = nn.Embedding(n_users, d_h)
        self.a = nn.Embedding(n_assets, d_id)
        d_in = d_id + (D_CLIP if use_clip else 0)
        self.mlp = nn.Sequential(nn.Linear(d_in, d_h), nn.ReLU(), nn.Linear(d_h, d_h))
    def forward(self, u, a, clip):
        x = self.a(a)
        if self.use_clip:
            x = torch.cat([x, clip], dim=1)
        return (self.u(u) * self.mlp(x)).sum(1)

CLIP_T = torch.tensor(clip_emb).to(device)

def fit(model, dfl, epochs=5, bs=4096, lr=1e-2):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss(reduction="none")
    u = torch.tensor(dfl.user.values).to(device)
    aa = torch.tensor(dfl.asset_id.values).to(device)
    y = torch.tensor(dfl.reward.values, dtype=torch.float32).to(device)
    # reward-based weighting: positives weighted by their interaction type's rho
    w = torch.tensor(dfl.itype.map(RHO).fillna(1.0).values, dtype=torch.float32).to(device)
    for _ in range(epochs):
        idx = torch.randperm(len(dfl), device=device)
        for i in range(0, len(dfl), bs):
            j = idx[i:i+bs]
            opt.zero_grad()
            loss = (lossf(model(u[j], aa[j], CLIP_T[aa[j]]), y[j]) * w[j]).mean()
            loss.backward(); opt.step()
    return model.eval()

## 6. Offline evaluation: inverse propensity scoring (IPS)

The most transferable idea in the paper. Judging a new policy on logs of the old policy is biased: the old policy over-shows its own favorites. Fix: on a randomized slice with **known, logged propensities**, reweight each observation by 1/propensity. The IPS value of a greedy policy:

`V(π) = mean over logs of [ 1{π picks the logged variant} · reward / propensity ]`

Unbiased because the propensities are known by construction — Netflix calls this the single biggest reason their offline numbers track online results.

In [ ]:
@torch.no_grad()
def ips_value(model, dfl):
    out = {}
    for ci in sorted(dfl.canvas.unique()):
        d = dfl[dfl.canvas == ci]
        parts = []
        for m in d.movieId.unique():
            ids = np.array([cand.loc[(m, k, ci)] for k in range(K)])
            us = d[d.movieId == m].user.unique()
            uu = torch.tensor(np.repeat(us, K)).to(device)
            ta = torch.tensor(np.tile(ids, len(us))).to(device)
            s = model(uu, ta, CLIP_T[ta]).reshape(len(us), K).argmax(1).cpu().numpy()
            parts.append(pd.DataFrame({"user": us, "movieId": m, "pick": s}))
        p = pd.concat(parts)
        dd = d.merge(p, on=["user", "movieId"])
        out[ci] = float(((dd.pick == dd.variant) * dd.reward / dd.prop).mean())
    return out

N_U, N_A = len(users), len(assets)
results = {}

# V2: one unified model, ID-only (no content)
results["V2 unified, ID-only"] = ips_value(
    fit(AssetModel(N_U, N_A, use_clip=False), train_log), explore_cold)

# V3: one unified model, ID + CLIP
m3 = fit(AssetModel(N_U, N_A, use_clip=True), train_log)
results["V3 unified, ID+CLIP"] = ips_value(m3, explore_cold)

# V1: five per-canvas models, each ID + CLIP
v1 = {}
for ci in CANVAS_VOL:
    mm = fit(AssetModel(N_U, N_A, use_clip=True), train_log[train_log.canvas == ci])
    v1.update(ips_value(mm, explore_cold[explore_cold.canvas == ci]))
results["V1 per-canvas, ID+CLIP"] = v1

# baseline: value of the random logging policy = plain mean reward
results["random policy"] = explore_cold.groupby("canvas").reward.mean().to_dict()

canvas_names = list(CANVASES.keys())
res_df = pd.DataFrame(results).rename(index=dict(enumerate(canvas_names)))
res_df.round(4)

In [ ]:
# the paper's headline chart: relative IPS lift vs random, per canvas
lift = res_df.div(res_df["random policy"], axis=0).drop(columns="random policy") - 1
ax = (lift * 100).plot.bar(figsize=(9, 4), rot=20)
ax.set_ylabel("IPS lift vs random policy (%)")
ax.set_title("Cold-start movies: only unified + CLIP (V3) wins")
ax.axhline(0, color="k", lw=0.8)
plt.tight_layout(); plt.show()

**Verified result (our run, 553 movies, seed 42):** V3 lifted every canvas — billboard +5%, vertical +9%, landscape +20%, short_panel +19%, panel +3% — while V2 (ID-only) and V1 (per-canvas + CLIP) bounced around zero (−5% to +7%, noise).

Same qualitative finding as Netflix's online A/B test: **neither ingredient works alone; only the combination does.**
- ID-only ≈ random on cold titles: a new asset is an unseen ID, nothing transfers.
- Per-canvas + CLIP: each sparse canvas has too few examples to learn how to *use* the image content.
- Unified + CLIP: rich canvases teach the shared model how image content maps to taste, and that mapping transfers to sparse canvases and unseen movies.

Exact numbers vary with seed and movie cap; the pattern (V3 ≫ V1 ≈ V2 ≈ 0) is the result.

## 7. The linear probe: a cheap gate for new embeddings

Before a full integration + A/B test, Netflix asks one cheap question: *from the content embedding alone, can a linear model predict which asset wins under an unpersonalized policy?* If yes, the embedding carries the semantic drivers of appeal. Fit on warm movies' debiased winners, test on cold movies.

In [ ]:
def winners(dfl, canvas=0):
    d = dfl[dfl.canvas == canvas]
    v = d.groupby(["movieId", "variant"]).apply(
        lambda g: (g.reward / g.prop).mean()).rename("v").reset_index()
    return v.loc[v.groupby("movieId").v.idxmax()][["movieId", "variant"]]

def probe_data(win, canvas=0):
    X, y = [], []
    for _, r in win.iterrows():
        for k in range(K):
            X.append(clip_emb[cand.loc[(r.movieId, k, canvas)]])
            y.append(int(k == r.variant))
    return np.array(X), np.array(y)

Xtr, ytr = probe_data(winners(explore_warm))
Xte, yte = probe_data(winners(explore_cold))
probe = LogisticRegression(max_iter=3000).fit(Xtr, ytr)
sc = probe.decision_function(Xte).reshape(-1, K)
acc = float((sc.argmax(1) == yte.reshape(-1, K).argmax(1)).mean())
print(f"probe winner accuracy on cold movies: {acc:.3f}  (random = {1/K:.3f})")

plt.figure(figsize=(4, 3.5))
plt.bar(["random", "linear probe\non CLIP"], [1/K, acc], color=["#999", "#2a7"])
plt.ylabel("winner prediction accuracy"); plt.title("The cheap gate")
plt.tight_layout(); plt.show()

**Verified result:** 0.355 vs 0.25 random (+42% relative). Modest in absolute terms — winners here are noisy (few impressions per movie) — but exactly the point of the gate: a linear model on the embedding alone beats chance, so the embedding carries appeal signal worth a full experiment.

## 8. Query-aware ranking, almost for free

CLIP puts text and images in one space, so a search-aware ranker is one extra term:

`score = α · personalization + (1 − α) · cos(CLIP_text(query), CLIP_image(asset))`

No new model — the same stored embeddings, one cosine at scoring time.

In [ ]:
@torch.no_grad()
def text_emb(q):
    t = tokenizer([q]).to(device)
    e = clip_model.encode_text(t)
    return (e / e.norm(dim=-1, keepdim=True)).cpu().numpy()[0]

def query_aware_top(query, u_idx, model, alpha=0.5, canvas=1, n=6):
    q = text_emb(query)
    ids = assets[assets.canvas == canvas].asset_id.values
    uu = torch.full((len(ids),), u_idx, dtype=torch.long).to(device)
    ta = torch.tensor(ids).to(device)
    with torch.no_grad():
        pers = model(uu, ta, CLIP_T[ta]).cpu().numpy()
    pers = (pers - pers.mean()) / (pers.std() + 1e-8)   # put both terms on one scale
    sim = clip_emb[ids] @ q
    sim = (sim - sim.mean()) / (sim.std() + 1e-8)
    return ids[np.argsort(-(alpha * pers + (1 - alpha) * sim))[:n]]

for q in ["a woman's face in close-up", "an explosion", "a cartoon animal"]:
    top6 = query_aware_top(q, u_idx=0, model=m3)
    fig, axes = plt.subplots(1, len(top6), figsize=(12, 2.6))
    for ax, i in zip(axes, top6):
        ax.imshow(asset_imgs[i]); ax.axis("off")
    fig.suptitle(f'query: "{q}"'); plt.tight_layout(); plt.show()

## 9. What this rebuild shows, and what it can't

**Shows (verified end to end):**
1. Content embeddings fix cold-start: an ID-only model is ~random on new titles; add CLIP and taste transfers from assets the model has seen to ones it hasn't.
2. Consolidation + content beat either alone (V3 ≫ V1 ≈ V2 ≈ 0). Netflix found the same thing online — only V3 moved their metrics. When a feature "doesn't help," look for a second blocking factor.
3. IPS with logged propensities is how you evaluate a new policy offline without an A/B test.
4. A linear probe is a cheap, honest gate for new embeddings.
5. Bonus, learned the hard way: raw CLIP vectors share a common direction (unrelated movies' cosine ≈ 0.28). Center before using cosines as a similarity signal, or differences drown.

**Can't show:** real effect sizes. Interactions are simulated (Assumptions 1–6), and the simulation is built from CLIP embeddings, which favors the CLIP model by construction. Netflix's evidence is a month-long A/B test; this notebook's contribution is the working mechanics.

**Not covered:** MediaFM / video previews (no public equivalent) — the obvious extension.

Source: [MAPS, Netflix TechBlog, Aug 2026](https://netflixtechblog.com/maps-netflixs-multimodal-asset-personalization-at-scale-32f96320785e). Code: [github.com/ZFayFay/netflix-maps-replication](https://github.com/ZFayFay/netflix-maps-replication)